# Semantic Similar-Ticket Retrieval

This notebook builds and evaluates a semantic retrieval system for
ResolveAI.

The objective is to retrieve historically similar customer-support
tickets for a new customer query using dense sentence embeddings.

The retrieval corpus contains only the leakage-safe training split.
Validation, challenge, and official test examples are excluded from
the retrieval index and are used only as retrieval queries.

This component will later provide contextual support examples for the
ResolveAI RAG resolution-generation pipeline.

In [1]:
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
SEED = 42

In [3]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print("Training corpus:", train_df.shape)
print("Validation queries:", validation_df.shape)

Training corpus: (18780, 9)
Validation queries: (4016, 9)


In [4]:
retrieval_corpus = (
    train_df[
        [
            "processed_instruction",
            "intent",
            "response"
        ]
    ]
    .copy()
)

validation_queries = (
    validation_df[
        [
            "processed_instruction",
            "intent"
        ]
    ]
    .copy()
)

### Load CPU Embedding Model

In [5]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\D_Drive\ResolveAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ribhu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

Embedding dimension: 384


C:\Users\ribhu\AppData\Local\Temp\ipykernel_41308\289703839.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


### Encoding a Query

In [7]:
sample_text = (
    retrieval_corpus[
        "processed_instruction"
    ]
    .iloc[0]
)

sample_embedding = (
    embedding_model.encode(
        sample_text
    )
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    sample_embedding[:10]
)

Embedding shape: (384,)
[-0.03027715  0.05177317  0.10175335  0.01067996 -0.02903386 -0.00467919
  0.03092369 -0.05523818  0.10852908  0.09587106]


### Test semantic similarity manually

In [8]:
example_sentences = [
    "where is my refund",
    "the money from my return has not arrived",
    "how can I change my account email",
    "my refund is still missing"
]

example_embeddings = (
    embedding_model.encode(
        example_sentences,
        normalize_embeddings=True
    )
)

similarities = cosine_similarity(
    [example_embeddings[0]],
    example_embeddings
)[0]

for sentence, similarity in zip(
    example_sentences,
    similarities
):
    print(
        f"{similarity:.4f}  {sentence}"
    )

1.0000  where is my refund
0.5982  the money from my return has not arrived
0.1961  how can I change my account email
0.8247  my refund is still missing
